In [2]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.labeling import add_returns
from src.volatility import add_close_to_close_volatility

#Load data
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

#Feature construction
df = add_returns(df)
df = add_close_to_close_volatility(df)

df = df.dropna(subset=["return", "vol_cc"]).reset_index(drop=True)

In [3]:
hmm_data = df[["return", "vol_cc"]].values #HMM inputs

# **Fit HMM Model**

In [4]:
from hmmlearn.hmm import GaussianHMM

hmm_model = GaussianHMM(
    n_components=3,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)

hmm_model.fit(hmm_data)

,n_components,3
,covariance_type,'full'
,min_covar,0.001
,startprob_prior,1.0
,transmat_prior,1.0
,means_prior,0
,means_weight,0
,covars_prior,0.01
,covars_weight,1
,algorithm,'viterbi'
,random_state,42


# **Predicting Hidden States**

In [5]:
states = hmm_model.predict(hmm_data)

df["state"] = states

In [7]:
df.head()

,Date,Open,High,Low,Close,Volume,return,vol_cc,state
0,2007-10-16,5670.649902,5708.350098,5578.450195,5668.049805,0,-0.000414,0.288382,2
1,2007-10-17,5658.899902,5658.899902,5107.299805,5559.299805,0,-0.019186,0.308638,2
2,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,0,-0.037469,0.331769,2
3,2007-10-19,5360.350098,5390.850098,5101.750000,5215.299805,0,-0.025360,0.350485,2
4,2007-10-22,5202.750000,5247.399902,5070.899902,5184.000000,0,-0.006002,0.348319,2


# **Map States -> Regime Labels**

In [8]:
state_vol = df.groupby("state")["vol_cc"].mean().sort_values()

state_mapping = {
    state_vol.index[0]: "Low",
    state_vol.index[1]: "Medium",
    state_vol.index[2]: "High"
}

df["hmm_regime"] = df["state"].map(state_mapping)

In [10]:
#Saving again
df[["Date", "hmm_regime"]].to_csv(
    "../data/processed/hmm_regimes_refit.csv",
    index=False
)

In [11]:
df.groupby("hmm_regime").agg(
    mean_return=("return","mean"),
    mean_vol=("vol_cc","mean"),
    count=("return","size")
)

,mean_return,mean_vol,count
hmm_regime,,,
High,0.000536,0.310212,1169
Low,-0.000106,0.125288,1663
Medium,0.000857,0.125325,1669


In [12]:
hmm_model.transmat_

array([[1.10667287e-05, 9.90662056e-01, 9.32687696e-03],
       [9.96167998e-01, 5.95292055e-05, 3.77247275e-03],
       [9.79775771e-03, 9.63254982e-03, 9.80569692e-01]])